In [ ]:
import numpy as np
import pandas as pd
from google.colab import files
files.upload()

# Create + Inspect + Basic Cleaning

In [ ]:
import pandas as pd
df = pd.read_csv('orders.csv')

print("HEAD:")
print(df.head(), "\n")

print("INFO:")
print(df.info(), "\n")

print("DESCRIBE:")
print(df.describe(include="all"), "\n")

# df['order_date'] = pd.to_datetime(df['order_date'], format='%m/%d/%Y', errors='coerce')
# not required to convert into datetime its already in that format

str_cols = df.select_dtypes(include='object').columns
df[str_cols] = df[str_cols].apply(lambda x: x.str.strip())

print("CLEANED DataFrame:")
print(df.head())


# Add Derived Columns

In [ ]:
df["gross_amount"] = df["quantity"] * df["unit_price"]

df["net_amount"] = df["gross_amount"] * (1 - df["discount_pct"] / 100)
threshold = 6000

df["is_high_value"] = df["net_amount"] > threshold

print("Data After adding gross_amount net_amount and is_high_value : \n")
print(df.head())


# Filtering + Multi-condition Queries

In [ ]:
categories = {"Electronics", "Fashion"}
X = 6000
N = 714

df["order_date"] = pd.to_datetime(df["order_date"])

cutoff_date = pd.Timestamp.today() - pd.Timedelta(days=N)
print(cutoff_date)

filtered_df = df[
    df["category"].isin(categories) &
    (df["net_amount"] >= X) &
    (df["order_date"] >= cutoff_date)
]

order_count = filtered_df.shape[0]
total_net_amount = filtered_df["net_amount"].sum()
print(order_count)
print(total_net_amount)


# GroupBy Aggregations


In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])

city_summary = (
    df.groupby("city")
      .agg(
          total_orders=("order_id", "count"),
          unique_customers=("customer_id", "nunique"),
          total_revenue=("net_amount", "sum"),
          avg_order_value=("net_amount", "mean"),
          max_order_date=("order_date", "max")
      )
      .sort_values("total_revenue", ascending=False)
      .head(10)
)

print(city_summary)

# Pivot Table Dashboard View

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])

df["month"] = df["order_date"].dt.to_period("M").astype(str)

# create pivot table
pivot_df = pd.pivot_table(
    df,
    index="month",
    columns="category",
    values="net_amount",
    aggfunc="sum",
    fill_value=0
)

pivot_df["Grand Total"] = pivot_df.sum(axis=1)

pivot_df["MoM Growth %"] = pivot_df["Grand Total"].pct_change() * 100

print("Pivot Table : \n")
print(pivot_df)


# Handling Missing Values

In [ ]:
cols_to_null = ["city", "payment_mode", "discount_pct"]

for col in cols_to_null:
    df.loc[df.sample(frac=0.1).index, col] = np.nan

# storing it to compare it later
missing_before = df[cols_to_null].isna().sum()
# print(missing_before)

categorical_cols = ["city", "payment_mode"]
df[categorical_cols] = df[categorical_cols].fillna("Unknown")

df["discount_pct"] = (
    df.groupby("category")["discount_pct"]
      .transform(lambda x: x.fillna(x.median()))
)

missing_after = df[cols_to_null].isna().sum()
# print(missing_after)

missing_summary = pd.DataFrame({
    "Before": missing_before,
    "After": missing_after
})

print(missing_summary)


# Joins / Merges (Customers + Orders)

In [ ]:

customers = pd.read_csv("customers.csv")

customers["signup_date"] = customers["signup_date"].str.replace("", "-", regex=False)

customers["signup_date"] = pd.to_datetime(customers["signup_date"])

df["order_date"] = pd.to_datetime(df["order_date"])

df2 = pd.merge(
    df,
    customers,
    on="customer_id",
    how="left"
)

segment_revenue = df2.groupby("segment")["net_amount"].sum().reset_index()
segment_revenue = segment_revenue.rename(columns={"net_amount": "total_revenue"})
# print(segment_revenue)


cutoff_date = pd.Timestamp.today() - pd.Timedelta(days=60)
df2["active_60d"] = df2["order_date"] >= cutoff_date

retention = (
    df2.groupby("segment")
      .agg(
          total_customers=("customer_id", "nunique"),
          active_customers=("active_60d", lambda x: x.groupby(df2.loc[x.index, "customer_id"]).any().sum())
      )
      .reset_index()
)
retention["retention_pct"] = retention["active_customers"] / retention["total_customers"] * 100

segment_summary = pd.merge(segment_revenue, retention, on="segment")

print(segment_summary)


# Window Functions (Intermediate)

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])

# Sort df per customer
df = df.sort_values(
    ["customer_id", "order_date"]
).reset_index(drop=True)

df["prev_order_date"] = (
    df
    .groupby("customer_id")["order_date"]
    .shift(1)
)

df["days_since_prev"] = (
    df["order_date"] - df["prev_order_date"]
).dt.days

df["rolling_3_order_avg"] = (
    df
    .groupby("customer_id")["net_amount"]
    .rolling(window=3, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)
# Identify customers with increasing average order value
# Simple heuristic:
# last rolling avg > first rolling avg

trend = (
    df
    .groupby("customer_id")["rolling_3_order_avg"]
    .agg(first_avg="first", last_avg="last")
    .reset_index()
)

increasing_customers = trend[
    trend["last_avg"] > trend["first_avg"]
]

print("Customers with increasing average order value:")
print(increasing_customers)

df.to_csv("df_with_window_metrics.csv", index=False)
increasing_customers.to_csv(
    "customers_with_increasing_aov.csv",
    index=False
)


# Outlier Detection + Capping (Intermediate)

In [ ]:
iqr_stats = (
    df
    .groupby("category")["net_amount"]
    .quantile([0.25, 0.75])
    .unstack()
    .rename(columns={0.25: "Q1", 0.75: "Q3"})
)

iqr_stats["IQR"] = iqr_stats["Q3"] - iqr_stats["Q1"]
iqr_stats["lower_bound"] = iqr_stats["Q1"] - 1.5 * iqr_stats["IQR"]
iqr_stats["upper_bound"] = iqr_stats["Q3"] + 1.5 * iqr_stats["IQR"]


df = df.merge(
    iqr_stats[["lower_bound", "upper_bound"]],
    left_on="category",
    right_index=True,
    how="left"
)

df["is_outlier_before"] = (
    (df["net_amount"] < df["lower_bound"]) |
    (df["net_amount"] > df["upper_bound"])
)

df["net_amount_capped"] = df["net_amount"].clip(
    lower=df["lower_bound"],
    upper=df["upper_bound"]
)

df["is_outlier_after"] = (
    (df["net_amount_capped"] < df["lower_bound"]) |
    (df["net_amount_capped"] > df["upper_bound"])
)

outlier_report = (
    df
    .groupby("category")
    .agg(
        outliers_before=("is_outlier_before", "sum"),
        outliers_after=("is_outlier_after", "sum"),
        total_rows=("net_amount", "count")
    )
    .reset_index()
)

print("Outlier report by category:")
print(outlier_report)

df.to_csv("orders_with_capped_outliers.csv", index=False)
outlier_report.to_csv("outlier_report_by_category.csv", index=False)


# Cohort Analysis (Intermediate)

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])

df["order_month"] = df["order_date"].dt.to_period("M")

df["cohort_month"] = (
    df
    .groupby("customer_id")["order_month"]
    .transform("min")
)


df["cohort_index"] = (
    df["order_month"] - df["cohort_month"]
).apply(lambda x: x.n)


cohort_data = (
    df
    .groupby(["cohort_month", "cohort_index"])["customer_id"]
    .nunique()
    .reset_index(name="active_customers")
)

cohort_table = cohort_data.pivot(
    index="cohort_month",
    columns="cohort_index",
    values="active_customers"
)


cohort_sizes = cohort_table[0]

retention = cohort_table.divide(
    cohort_sizes, axis=0
)

retention = retention.round(4) * 100

retention.index = retention.index.astype(str)
retention.columns = [
    f"M{int(col)}" for col in retention.columns
]


print("Retention Cohort Table (%):")
print(retention)
